# 📝 Module 2.2: Logging Basics — Parameters, Metrics & Tags

**Goal**: Master the core logging API — the foundation of everything you'll do with MLFlow.

---

## Setup

In [1]:
import mlflow
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# MLflow Database Configuration
# All modules share a centralized SQLite database at the project root
import os

_db_path = os.path.normpath(
    os.path.join(os.getcwd(), "..", "mlflow.db")
).replace(chr(92), "/")
mlflow.set_tracking_uri(f"sqlite:///{_db_path}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

# Set our experiment
mlflow.set_experiment("02_logging_basics")
print("✅ Experiment set!")

Tracking URI: sqlite:///c:/Users/sujat/projects/MLFlow_Learn/mlflow.db
✅ Experiment set!


## Load Data

We'll use the **Wine dataset** from sklearn — this will be our go-to dataset for the rest of the course.

In [2]:
# Load Wine dataset
wine = load_wine()
X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, test_size=0.2, random_state=42
)

print(f"Features: {wine.feature_names}")
print(f"Classes: {list(wine.target_names)}")
print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")

Features: ['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']
Classes: [np.str_('class_0'), np.str_('class_1'), np.str_('class_2')]
Train: 142 samples | Test: 36 samples


---

## 1. Logging Parameters

**Parameters** are the input settings for your experiment — hyperparameters, data processing choices, etc.

### Rules:
- Parameters are **immutable** — once logged, they can't be changed
- Values are stored as **strings** (MLFlow converts automatically)
- Use `log_param()` for one, `log_params()` for multiple

In [3]:
# Method 1: log_param() — one at a time
with mlflow.start_run(run_name="param_demo_single"):
    
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("test_size", 0.2)
    
    # Train and log a metric so the run is meaningful
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    accuracy = accuracy_score(y_test, model.predict(X_test))
    mlflow.log_metric("accuracy", accuracy)
    
    print(f"✅ Logged individual params. Accuracy: {accuracy:.4f}")

✅ Logged individual params. Accuracy: 1.0000


In [4]:
# Method 2: log_params() — dictionary (PREFERRED for cleaner code)
with mlflow.start_run(run_name="param_demo_dict"):
    
    params = {
        "model_type": "RandomForest",
        "n_estimators": 200,
        "max_depth": 10,
        "min_samples_split": 5,
        "test_size": 0.2,
        "random_state": 42,
    }
    
    mlflow.log_params(params)
    
    model = RandomForestClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_split=params["min_samples_split"],
        random_state=params["random_state"]
    )
    model.fit(X_train, y_train)
    accuracy = accuracy_score(y_test, model.predict(X_test))
    mlflow.log_metric("accuracy", accuracy)
    
    print(f"✅ Logged params dict. Accuracy: {accuracy:.4f}")

✅ Logged params dict. Accuracy: 1.0000


### 💡 Best Practice: Log Everything That Could Affect Results

Don't just log model hyperparameters — also log:
- Data processing choices (scaling method, feature selection)
- Dataset info (train/test split ratio, data version)
- Runtime info (random seed, hardware)

---

## 2. Logging Metrics

**Metrics** are numeric values that measure model performance.

### Rules:
- Metrics are **numeric** (int or float)
- You can log the **same metric key multiple times** with different `step` values (for training curves)
- Use `log_metric()` for one, `log_metrics()` for multiple

In [5]:
# Log multiple metrics at once
with mlflow.start_run(run_name="metrics_demo-Hola"):
    
    # Log params
    params = {"model_type": "RandomForest", "n_estimators": 150, "max_depth": 7}
    mlflow.log_params(params)
    
    # Train
    model = RandomForestClassifier(n_estimators=150, max_depth=7, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Calculate multiple metrics
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_weighted": precision_score(y_test, y_pred, average='weighted'),
        "recall_weighted": recall_score(y_test, y_pred, average='weighted'),
        "f1_weighted": f1_score(y_test, y_pred, average='weighted'),
        "num_features": X_train.shape[1],
        "num_training_samples": X_train.shape[0],
    }
    
    # Log all metrics at once
    mlflow.log_metrics(metrics)
    
    print("✅ Metrics logged:")
    for k, v in metrics.items():
        print(f"   {k}: {v:.4f}")

✅ Metrics logged:
   accuracy: 1.0000
   precision_weighted: 1.0000
   recall_weighted: 1.0000
   f1_weighted: 1.0000
   num_features: 13.0000
   num_training_samples: 142.0000


### Step-Based Metric Logging (Training Curves)

For iterative training, you can log metrics at each **step** (epoch/iteration) to create training curves. This is especially useful for deep learning, but let's simulate it with a simple example.

In [6]:
# Simulate step-based logging (like tracking loss per epoch)
with mlflow.start_run(run_name="step_metrics_demo"):
    
    mlflow.log_param("model_type", "simulated_training")
    mlflow.log_param("epochs", 20)
    mlflow.log_param("learning_rate", 0.01)
    
    # Simulate a training loop with decreasing loss and increasing accuracy
    np.random.seed(42)
    for epoch in range(20):
        # Simulate metrics that improve over time
        train_loss = 1.0 * np.exp(-0.15 * epoch) + np.random.normal(0, 0.02)
        val_loss = 1.0 * np.exp(-0.12 * epoch) + np.random.normal(0, 0.03)
        train_acc = 1.0 - 0.5 * np.exp(-0.15 * epoch) + np.random.normal(0, 0.01)
        val_acc = 1.0 - 0.55 * np.exp(-0.12 * epoch) + np.random.normal(0, 0.015)
        
        # Log with step parameter — creates a time series!
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("train_accuracy", train_acc, step=epoch)
        mlflow.log_metric("val_accuracy", val_acc, step=epoch)
    
    print("✅ Step-based metrics logged!")
    print("   Open MLFlow UI → click this run → see the training curves!")

✅ Step-based metrics logged!
   Open MLFlow UI → click this run → see the training curves!


---

## 3. Setting Tags

**Tags** are string key-value pairs for organizing and annotating runs.

### Common uses:
- `team`, `author` — who ran this
- `purpose` — why this run exists (baseline, experiment, production)
- `data_version` — which data version was used
- `notes` — free-form notes about the run

In [7]:
with mlflow.start_run(run_name="tags_demo"):
    
    # Set individual tags
    mlflow.set_tag("author", "sujat")
    mlflow.set_tag("purpose", "baseline")
    mlflow.set_tag("data_version", "v1.0")
    
    # Set multiple tags at once
    mlflow.set_tags({
        "model_family": "tree-based",
        "feature_engineering": "none",
        "notes": "First baseline with default hyperparameters"
    })
    
    # Train and log
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    mlflow.log_param("n_estimators", 100)
    mlflow.log_metric("accuracy", accuracy_score(y_test, model.predict(X_test)))
    
    print("✅ Tags set!")
    print("   Check the MLFlow UI → click the run → see tags in the sidebar")

✅ Tags set!
   Check the MLFlow UI → click the run → see tags in the sidebar


### 💡 System Tags

MLFlow automatically sets some tags with the `mlflow.` prefix:

| Tag | Description |
|-----|-------------|
| `mlflow.runName` | Name of the run |
| `mlflow.user` | User who started the run |
| `mlflow.source.type` | How the run was started (NOTEBOOK, LOCAL, etc.) |
| `mlflow.source.name` | Source file name |

---

## 4. Putting It All Together — Complete Example

Let's do a proper experiment comparing **two model types**, logging everything properly.

In [8]:
# Define experiments to compare
experiments = [
    {
        "run_name": "random_forest_baseline",
        "model_type": "RandomForest",
        "model": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
        "params": {"n_estimators": 100, "max_depth": 5},
    },
    {
        "run_name": "random_forest_deep",
        "model_type": "RandomForest",
        "model": RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42),
        "params": {"n_estimators": 200, "max_depth": 15},
    },
    {
        "run_name": "logistic_regression_baseline",
        "model_type": "LogisticRegression",
        "model": LogisticRegression(max_iter=1000, C=1.0, random_state=42),
        "params": {"max_iter": 1000, "C": 1.0},
    },
    {
        "run_name": "logistic_regression_regularized",
        "model_type": "LogisticRegression",
        "model": LogisticRegression(max_iter=1000, C=0.1, random_state=42),
        "params": {"max_iter": 1000, "C": 0.1},
    },
]

print("🔄 Running experiments...\n")

for exp in experiments:
    with mlflow.start_run(run_name=exp["run_name"]):
        
        # Log parameters
        mlflow.log_params(exp["params"])
        mlflow.log_param("model_type", exp["model_type"])
        
        # Tags
        mlflow.set_tags({
            "author": "sujat",
            "purpose": "model_comparison",
            "model_family": exp["model_type"]
        })
        
        # Train
        model = exp["model"]
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Log metrics
        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, average='weighted'),
            "recall": recall_score(y_test, y_pred, average='weighted'),
            "f1": f1_score(y_test, y_pred, average='weighted'),
        }
        mlflow.log_metrics(metrics)
        
        print(f"  ✅ {exp['run_name']:.<40} accuracy={metrics['accuracy']:.4f}")

print("\n🎉 All experiments complete! Compare them in the MLFlow UI.")

🔄 Running experiments...

  ✅ random_forest_baseline.................. accuracy=1.0000
  ✅ random_forest_deep...................... accuracy=1.0000


c:\Users\sujat\projects\MLFlow_Learn\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  ✅ logistic_regression_baseline............ accuracy=0.9722
  ✅ logistic_regression_regularized......... accuracy=1.0000

🎉 All experiments complete! Compare them in the MLFlow UI.


c:\Users\sujat\projects\MLFlow_Learn\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 🔑 Key Takeaways

1. **`log_params()`** for input configuration — use a dict for clean code
2. **`log_metrics()`** for output measurements — use `step` for training curves
3. **`set_tags()`** for metadata — annotate runs with context (author, purpose, notes)
4. **Always name your runs** with `run_name` parameter — makes the UI much easier to navigate
5. Log **everything that could affect results** — you'll thank yourself later

---

## ➡️ Next: `03_logging_artifacts.ipynb` — Log files, plots, and datasets